# MeguvizCAC Turbo v3 Studio — Colab G4 (RTX PRO 6000 Blackwell 96 GB)

Structured multi-cell foreground notebook designed for **zero idle-disconnect risk** (NO `nohup` background daemons) and **maximum GPU utilization**:
- **Content-Hash-Keyed Rust (`meguviz_native`) Wheel Cache**: computes `sha8` from `meguviz_native/Cargo.toml` + `meguviz_native/src/lib.rs` (`~3s` install from Drive when unchanged; automatic `~9s` `maturin build --release` when Rust source changes)
- **Turbo v3 Trainer**: `torch.compile(dynamic=True)` + `BF16 AMP` + `FusedAux4` + `eval_batch_size=512` + GPU `_foreach_lerp_` EMA + Rust `kadane_focal_window`
- **Primary Foreground Track (`v1.6.2`: `v1.6.2-S` & `v1.6.2-PUC`) + Optional `v1.6.1-BW` & CUDA MPS Parallel Packing**

In [ ]:
# Cell 1: Mount Drive, Pull Latest Code, Content-Hash-Keyed Rust `meguviz_native` Wheel Cache
import hashlib, os, shutil, subprocess, sys, tempfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_DIR = Path('/content/MeguvizCAC')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jichuo1/MeguvizCAC.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)

subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)

# Content-hash-keyed Rust wheel cache (Cargo.toml + src/lib.rs -> sha8)
h = hashlib.sha256()
for rel in ('meguviz_native/Cargo.toml', 'meguviz_native/src/lib.rs'):
    h.update((REPO_DIR / rel).read_bytes())
rust_sha8 = h.hexdigest()[:8]
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
wheel_cache_dir = Path('/content/drive/MyDrive/MeguvizCAC/wheels') / rust_sha8
wheel_cache_dir.mkdir(parents=True, exist_ok=True)
sha_marker = Path('/tmp/.meguviz_rs_sha8')

need_install = not (sha_marker.is_file() and sha_marker.read_text().strip() == rust_sha8)
if not need_install:
    try:
        import _meguviz_rs
        assert hasattr(_meguviz_rs, 'extract_post_bpe_all_rs') and hasattr(_meguviz_rs.RustOOFCharLMSet, 'kadane_focal_window')
    except Exception:
        need_install = True

if need_install:
    whls = sorted(wheel_cache_dir.glob(f'meguviz_native-*-{py_tag}-*x86_64.whl'), key=lambda p: p.stat().st_mtime, reverse=True)
    if whls:
        print(f'Installing cached Rust wheel (sha8={rust_sha8}):', whls[0])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(whls[0])], check=True)
    else:
        print(f'Building Rust meguviz_native wheel (sha8={rust_sha8}) & caching to Drive...')
        if subprocess.run(['bash', '-lc', 'command -v cargo'], capture_output=True).returncode != 0:
            subprocess.run(['bash', '-lc', 'curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'maturin'], check=True)
        with tempfile.TemporaryDirectory() as tmpdir:
            subprocess.run([
                'bash', '-lc',
                f'export PATH="$HOME/.cargo/bin:$PATH" && maturin build --release --manifest-path {REPO_DIR}/meguviz_native/Cargo.toml -i {sys.executable} -o {tmpdir}'
            ], check=True)
            built = sorted(Path(tmpdir).glob(f'meguviz_native-*-{py_tag}-*x86_64.whl'), key=lambda p: p.stat().st_mtime, reverse=True)
            assert built, f'No wheel produced in {tmpdir}'
            cached_whl = wheel_cache_dir / built[0].name
            shutil.copy2(built[0], cached_whl)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(cached_whl)], check=True)
    sha_marker.write_text(rust_sha8)

import _meguviz_rs
assert hasattr(_meguviz_rs, 'extract_post_bpe_all_rs') and hasattr(_meguviz_rs.RustOOFCharLMSet, 'kadane_focal_window')
print(f'Rust _meguviz_rs READY (sha8={rust_sha8}):', _meguviz_rs.__file__)

In [ ]:
# Cell 2: Turbo v3 Parameters (edit here — applied to foreground training cells below)
TURBO_ENV = {
    'BS': '256',
    'EVAL_BS': '512',
    'SEED': '20261220',
    'USE_COMPILE': '1',
    'COMPILE_MODE': 'default',
    'COMPILE_DYNAMIC': '1',
    'AMP_BF16': '1',
    'FUSE_AUX': '1',
    'PREFETCH_WORKERS': '2',
    'RUN_TRACK_S': '1',
    'RUN_TRACK_PUC': '1',
}
print('Turbo v3 parameters configured:', TURBO_ENV)

In [ ]:
# Cell 3: Primary Foreground Run — v1.6.2 (v1.6.2-S + v1.6.2-PUC with Turbo v3)
import os, subprocess

env = {**os.environ, **TURBO_ENV}
proc = subprocess.run(
    ['bash', '/content/MeguvizCAC/meguviz_cac/run_v162_colab.sh'],
    cwd='/content/MeguvizCAC',
    env=env,
    check=True,
)
print('v1.6.2 finished with exit code:', proc.returncode)

In [ ]:
# Cell 4A (OPTIONAL): Foreground v1.6.1-BW Runner (Turbo v3)
import os, subprocess

env = {**os.environ, **TURBO_ENV, 'P2_STEPS': '2800', 'P2_LR': '5.5e-4'}
subprocess.run(['bash', '/content/MeguvizCAC/meguviz_cac/run_v161_bw_colab.sh'], cwd='/content/MeguvizCAC', env=env, check=True)

In [ ]:
# Cell 4B (OPTIONAL): 96 GB Blackwell CUDA MPS Parallel Packing (v1.6.2-S + v1.6.2-PUC simultaneously)
# Prerequisite: shared_r0_v162_frozen.pt already exists in Drive (or run Track S first).
# Runs strictly in foreground — waits for both processes while streaming live status every 30s.
import os, subprocess, time

env_base = {**os.environ, **TURBO_ENV}
print('Starting CUDA MPS daemon for multi-process 96 GB Blackwell packing...')
subprocess.run(['bash', '-lc', 'nvidia-cuda-mps-control -d || true'])

p_s = subprocess.Popen(['bash', '/content/MeguvizCAC/meguviz_cac/run_v162_colab.sh'], cwd='/content/MeguvizCAC', env={**env_base, 'RUN_TRACK_S': '1', 'RUN_TRACK_PUC': '0'})
p_puc = subprocess.Popen(['bash', '/content/MeguvizCAC/meguviz_cac/run_v162_colab.sh'], cwd='/content/MeguvizCAC', env={**env_base, 'RUN_TRACK_S': '0', 'RUN_TRACK_PUC': '1'})

t0 = time.time()
while p_s.poll() is None or p_puc.poll() is None:
    el = int(time.time() - t0)
    print(f'[{el:4d}s] Track-S={"RUN" if p_s.poll() is None else p_s.returncode} | Track-PUC={"RUN" if p_puc.poll() is None else p_puc.returncode}', flush=True)
    time.sleep(30)

print('Parallel MPS finish codes:', p_s.returncode, p_puc.returncode)
assert p_s.returncode == 0 and p_puc.returncode == 0

In [ ]:
# Cell 5: Final Scoreboard & Drive Artifact Verification (v1.6.2-S vs v1.6.2-PUC)
import json
from pathlib import Path

drv = Path('/content/drive/MyDrive/MeguvizCAC/v162_production')
cmp = drv / 'v162_comparison.json'
if cmp.is_file():
    print(json.dumps(json.loads(cmp.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
for sub in ('v1.6.2-S', 'v1.6.2-PUC'):
    d = drv / sub
    if d.is_dir():
        print(f'\n=== {d} ===')
        for f in sorted(d.iterdir()):
            if f.is_file():
                print(f'  {f.name:52s} {f.stat().st_size / 1e6:7.2f} MB')